Question 1

<small>To show my ability to use Git in Question 1, I created a gitHub repository for ASTR4004A2 and then within that added a README.md file to the main branch and connected the repository on motley to my GitHub. I created a branch to work on for Q1 and Q2 ADQL and then created a Jupyter Notebook file, commited that and then merged the branch and those changes back into my main. </small>

Question 2

In [2]:
from astroquery.gaia import Gaia

Part 1

<small> For part 1 of Q2 below, I used ASTROQUERY to write a query that returned all stars in the Gaia DR3 table within 1 degree of M67 and G\<14 and crossmatched them with 2MASS. The number of returned stars was 1018.</small>

In [7]:
query= f"""
SELECT gaia.*, 
    tmass.*, 
    xmatch.angular_distance
FROM gaiadr3.gaia_source AS gaia
JOIN gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch 
    USING (source_id)
JOIN gaiadr3.tmass_psc_xsc_join AS xjoin 
    USING (clean_tmass_psc_xsc_oid)
JOIN gaiadr1.tmass_original_valid AS tmass 
    ON xjoin.original_psc_source_id = tmass.designation
WHERE 
    1 = CONTAINS(
        POINT('ICRS', gaia.ra, gaia.dec),
        CIRCLE('ICRS', 132.825, 11.8, 1)
    )
    AND gaia.phot_g_mean_mag < 14"""

job= Gaia.launch_job_async(query)
result= job.get_results()
print('Number of Stars Returned:', len(result))

INFO: Query finished. [astroquery.utils.tap.core]
2869698913.py: Number of Stars Returned: 1018


Part 2

<small> For part 2 of Q2 below, I identified the stars with bad 2MASS photometry and non-positive Gaia parallax values and filtered these out of the sample returned in Part 1 above. 988 stars remained.</small>

In [ ]:
#First Identify Number of stars with bad 2MASS Quality
bad_2mass=result[result['ph_qual']!='AAA']
print('Number of Stars with Bad 2MASS Photometry', len(bad_2mass))
#Then identify number of stars with non-positive Gaia Parallax
non_pos_parallax=result[result['parallax']<=0]
print('Number of Stars with Non-positive Gaia Parallax', len(non_pos_parallax))
#Now to filter these stars out
clean=result[(result['ph_qual']=='AAA') & (result['parallax']>0)]
print('Number of Stars Remaining after Quality Cuts:', len(clean) )
#Check te number removed 
removed=result[~((result['ph_qual']=='AAA') & (result['parallax']>0))]
print('Number of Stars Removed due to Quality Cuts:', len(removed) )

Part 3: Two-Panel Figure

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2)= plt.subplots(1, 2, figsize=(12,5))
#First panel is Gaia BP-RP versus magnitude
ax1.scatter(clean['phot_g_mean_mag'], clean['bp_rp'])
ax1.set_title('Plot of Gaia BP-RP versus G magnitude')
ax1.set_ylabel('Gaia BP-RP')
ax1.set_xlabel('Gaia Absolute G magnitude')
#Second panel with 2MASS J-Ks versus apparent H magnitude
J_Ks=clean['j_m']-clean['ks_m']
ax2.scatter(clean['h_m'], J_Ks)
ax2.set_title('Plot of 2MASS J-Ks versus Apparent H magnitude')
ax2.set_ylabel('2MASS J-Ks')
ax2.set_xlabel('Apparent H magnitude')
#To save it
#mkdir -p fig
plt.savefig("fig/Q2Part3Gaia2MASSPlots.png", dpi=300, bbox_inches='tight')
#git add fig/plot
#git commit -m 
#git push origin Q1Q2ADQL

Part 4

<small> According to the AAT instruments website (https://aat.anu.edu.au/science/instruments/current/HERMES), 2dF which HERMES is built upon allows te acquisition of up to 392 simultaneous spectra in a two dgree field of sky. Thus, te fact that this 1 degree field of sky contains over 900 objects, there would most likely be too many objects than the fibre usage of HERMES allows. Thus, the only feasible way this may be enabled is if the field was broken up into three separate observations with the fibres positioned on different stars each time or if a subsample of about 300 stars was selected and observed. </small>